Download [dataset](https://www.kaggle.com/competitions/platesv2)

Place it in `/data`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.random.manual_seed(42)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
learning_rate = 1e-3
batch_size = 64
epochs = 1000

device

in test: *Dirty* - $392$, *Clean* - $208$

In [ ]:
import os
import cv2
from torch.utils.data import Dataset, DataLoader

class TrainDataset(Dataset):
    def __init__(self, path):
        self.data = []
        self.labels = []
        for file in os.listdir(os.join(path, 'clean')):
            self.data.append(cv2.imread(os.path.join(path, 'clean', file)))
            self.labels.append(0)
        for file in os.listdir(os.join(path, 'dirty')):
            self.data.append(cv2.imread(os.path.join(path, 'dirty', file)))
            self.labels.append(1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
import torch
import torch.nn as nn

backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')

for param in backbone.parameters():
    param.requires_grad = False

class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.head(features)

model = DINOv2Classifier(backbone)

backbone_params = list(model.backbone.parameters())
head_params = list(model.head.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-5, "weight_decay": 0.01},
        {"params": head_params, "lr": 1e-3, "weight_decay": 0.0},
    ]
)